# DimASR: Dimensional Aspect Sentiment Regression - Laptop Domain

**SemEval-2026 Task 3 - Track A: Subtask 1**

This notebook implements the DimASR task which predicts valence-arousal (VA) scores for given aspects in review text.
- **Valence**: Emotional positivity (1.00=very negative, 5.00=neutral, 9.00=very positive)
- **Arousal**: Emotional intensity (1.00=calm, 9.00=excited/intense)

Output format: `valence#arousal` (e.g., "7.12#6.88")

## Libraries

In [1]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except:
    IN_COLAB = False

In [2]:
if IN_COLAB:
    !pip install transformers datasets evaluate sentencepiece scipy

In [3]:
!pip install pandas


In [4]:
import os
import json
import torch
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

if IN_COLAB:
    root_path = 'Enter drive path'
else:
    root_path = '/root/ITATA'

use_mps = True if torch.backends.mps.is_built() else False
os.chdir(root_path)
print(f"Working directory: {os.getcwd()}")
print(f"MPS available: {use_mps}")
print(f"CUDA available: {torch.cuda.is_available()}")

Working directory: /root/ITATA
MPS available: False
CUDA available: True


In [5]:
from Imports.data_prep import DatasetLoader
from Imports.utils import T5Generator
from instructions import InstructionsHandler

## Configuration

In [6]:
# Task configuration
task_name = 'dimasr'
experiment_name = 'laptop_eng_v1'
model_checkpoint = 'allenai/tk-instruct-base-def-pos'  # or 'google/flan-t5-base'

# Data paths
train_file = './eng_laptop_train_alltasks.jsonl'
dev_file = './eng_laptop_dev_task1.jsonl'

# Model output path
model_out_path = os.path.join('./Models', task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Experiment Name:', experiment_name)
print('Model output path:', model_out_path)

Experiment Name: laptop_eng_v1
Model output path: ./Models/dimasr/allenaitk-instruct-base-def-pos-laptop_eng_v1


## Load Data

In [7]:
# Load JSONL training data
train_df = DatasetLoader.load_jsonl_data(train_file)
dev_df = DatasetLoader.load_jsonl_data(dev_file)

print(f"Training records: {len(train_df)}")
print(f"Dev records: {len(dev_df)}")
print(f"\nTraining columns: {train_df.columns.tolist()}")
print(f"Dev columns: {dev_df.columns.tolist()}")

Training records: 4076
Dev records: 200

Training columns: ['ID', 'Text', 'Quadruplet']
Dev columns: ['ID', 'Text', 'Aspect']


In [8]:
# Display sample data
print("Sample training record:")
print(train_df.iloc[0])

Sample training record:
ID                                            laptop_quad_dev_1
Text          this unit is ` ` pretty ` ` and stylish , so m...
Quadruplet    [{'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN...
Name: 0, dtype: object


## Setup Instructions

In [9]:
# Initialize instruction handler
instruct_handler = InstructionsHandler()

# Load instruction set (use set1 for basic, set2 for extended examples, set3 for most examples)
instruct_handler.load_instruction_set2()

print("DimASR Instructions loaded:")
print(f"- BOS instruction length: {len(instruct_handler.dimasr['bos_instruct1'])} chars")
print(f"- Delimiter: '{instruct_handler.dimasr['delim_instruct']}'")
print(f"- EOS: '{instruct_handler.dimasr['eos_instruct']}'")

DimASR Instructions loaded:
- BOS instruction length: 1155 chars
- Delimiter: ' The aspect is '
- EOS: '.
output:'


## Format Training Data

In [10]:
# Create data loader and format training data
loader = DatasetLoader(train_df_id=None, test_df_id=None)

# Format training data (expand Quadruplet to individual rows)
train_formatted = loader.create_data_in_dimasr_format(
    train_df,
    text_col='Text',
    quadruplet_col='Quadruplet',
    bos_instruction=instruct_handler.dimasr['bos_instruct1'],  # bos_instruct1 for laptop domain
    delim_instruction=instruct_handler.dimasr['delim_instruct'],
    eos_instruction=instruct_handler.dimasr['eos_instruct'],
    is_train=True
)

print(f"Expanded training samples: {len(train_formatted)}")
print(f"\nColumns: {train_formatted.columns.tolist()}")

Expanded training samples: 5773

Columns: ['ID', 'text', 'labels', 'aspect', 'original_text']


In [11]:
# Display sample formatted data
print("Sample formatted input (first 500 chars):")
print(train_formatted['text'].iloc[0][:500])
print("\n...")
print(f"\nLabel (VA): {train_formatted['labels'].iloc[0]}")
print(f"Aspect: {train_formatted['aspect'].iloc[0]}")

Sample formatted input (first 500 chars):
Definition: The output will be the valence and arousal scores for the given aspect in the input text. Valence measures emotional positivity (1.00=very negative, 5.00=neutral, 9.00=very positive). Arousal measures emotional intensity (1.00=calm, 9.00=excited/intense). Output format: valence#arousal with values from 1.00 to 9.00 rounded to two decimal places.
Positive example 1-
input: I charge it at night and skip taking the cord with me because of the good battery life. The aspect is battery lif

...

Label (VA): 7.12#7.12
Aspect: unit


## Train/Validation Split

In [12]:
# Split into training and validation sets
train_split, val_split = train_test_split(train_formatted, test_size=0.1, random_state=42)

# Update loader with split data
loader.train_df_id = train_split.reset_index(drop=True)
loader.val_df_id = val_split.reset_index(drop=True)

print(f"Training samples: {len(train_split)}")
print(f"Validation samples: {len(val_split)}")

Training samples: 5195
Validation samples: 578


## Initialize Model

In [13]:
# Create T5 Generator
t5_exp = T5Generator(model_checkpoint)
print(f"Model loaded: {model_checkpoint}")
print(f"Device: {t5_exp.device}")

Model loaded: allenai/tk-instruct-base-def-pos
Device: cuda


## Tokenize Dataset

In [14]:
# Tokenize datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(
    t5_exp.tokenize_function_inputs
)

print(f"Tokenized train samples: {len(id_tokenized_ds['train'])}")
print(f"Tokenized validation samples: {len(id_tokenized_ds['validation'])}")

Map:   0%|          | 0/5195 [00:00<?, ? examples/s]

Map:   0%|          | 0/578 [00:00<?, ? examples/s]

Tokenized train samples: 5195
Tokenized validation samples: 578


## Training

In [15]:
# Training arguments
training_args = {
    'output_dir': model_out_path,
    'evaluation_strategy': 'epoch',
    'learning_rate': 5e-5,
    'lr_scheduler_type': 'cosine',
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'push_to_hub': False,
    'eval_accumulation_steps': 1,
    'predict_with_generate': True,
    'use_mps_device': use_mps
}

print("Training configuration:")
for k, v in training_args.items():
    print(f"  {k}: {v}")

Training configuration:
  output_dir: ./Models/dimasr/allenaitk-instruct-base-def-pos-laptop_eng_v1
  evaluation_strategy: epoch
  learning_rate: 5e-05
  lr_scheduler_type: cosine
  per_device_train_batch_size: 8
  per_device_eval_batch_size: 16
  num_train_epochs: 5
  weight_decay: 0.01
  warmup_ratio: 0.1
  save_strategy: epoch
  load_best_model_at_end: True
  metric_for_best_model: eval_loss
  greater_is_better: False
  push_to_hub: False
  eval_accumulation_steps: 1
  predict_with_generate: True
  use_mps_device: False


In [16]:
# Train the model
model_trainer = t5_exp.train(id_tokenized_ds, **training_args)

Trainer device: cuda:0

Model training started ....


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,1.597800,1.139936
2,1.167900,1.041330
3,1.084200,0.995939
4,1.014300,0.987732
5,1.000200,0.988238


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


## Inference & Evaluation

In [17]:
# Load trained model for inference
t5_exp = T5Generator(model_out_path)
print(f"Loaded trained model from: {model_out_path}")

Loaded trained model from: ./Models/dimasr/allenaitk-instruct-base-def-pos-laptop_eng_v1


In [18]:
# Re-tokenize for inference
id_ds, id_tokenized_ds, _, _ = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get predictions on validation set
val_pred = t5_exp.get_labels(id_tokenized_ds, sample_set='validation', batch_size=16)
val_true = [label.strip() for label in id_ds['validation']['labels']]

print(f"Generated {len(val_pred)} predictions")

Map:   0%|          | 0/5195 [00:00<?, ? examples/s]

Map:   0%|          | 0/578 [00:00<?, ? examples/s]

Model loaded to:  cuda


100%|██████████| 37/37 [00:06<00:00,  6.11it/s]

Generated 578 predictions


In [19]:
# Calculate metrics
metrics = t5_exp.get_metrics_regression(val_true, val_pred)

print("\n" + "="*50)
print("VALIDATION METRICS")
print("="*50)
print(f"RMSE_VA (Official): {metrics['RMSE_VA']:.4f}")
print(f"PCC Valence:        {metrics['PCC_V']:.4f}")
print(f"PCC Arousal:        {metrics['PCC_A']:.4f}")
print(f"PCC Average:        {metrics['PCC_avg']:.4f}")
print(f"RMSE Valence:       {metrics['RMSE_V']:.4f}")
print(f"RMSE Arousal:       {metrics['RMSE_A']:.4f}")
print("="*50)


VALIDATION METRICS
RMSE_VA (Official): 1.3610
PCC Valence:        0.8427
PCC Arousal:        0.6119
PCC Average:        0.7273
RMSE Valence:       1.0067
RMSE Arousal:       0.9159


In [20]:
# Display sample predictions
print("\nSample Predictions vs Ground Truth:")
print("-" * 60)
for i in range(min(10, len(val_pred))):
    pred_formatted = t5_exp.parse_va_prediction(val_pred[i])
    print(f"True: {val_true[i]:12} | Pred: {val_pred[i]:12} | Formatted: {pred_formatted}")


Sample Predictions vs Ground Truth:
------------------------------------------------------------
True: 7.00#7.00    | Pred: 7.50#7.67    | Formatted: 7.50#7.67
True: 7.17#6.83    | Pred: 7.50#7.67    | Formatted: 7.50#7.67
True: 8.12#8.25    | Pred: 7.50#7.67    | Formatted: 7.50#7.67
True: 7.25#7.50    | Pred: 7.50#7.67    | Formatted: 7.50#7.67
True: 5.00#5.00    | Pred: 6.50#6.50    | Formatted: 6.50#6.50
True: 6.75#6.50    | Pred: 6.50#6.50    | Formatted: 6.50#6.50
True: 3.30#6.30    | Pred: 4.50#5.25    | Formatted: 4.50#5.25
True: 1.83#8.00    | Pred: 2.50#7.62    | Formatted: 2.50#7.62
True: 4.25#4.50    | Pred: 4.50#5.25    | Formatted: 4.50#5.25
True: 7.50#7.67    | Pred: 7.50#7.67    | Formatted: 7.50#7.67


## Generate Predictions for Dev Set (Submission)

In [21]:
# Format dev data for inference (no labels)
dev_formatted = loader.create_data_in_dimasr_format(
    dev_df,
    text_col='Text',
    aspect_col='Aspect',
    bos_instruction=instruct_handler.dimasr['bos_instruct1'],
    delim_instruction=instruct_handler.dimasr['delim_instruct'],
    eos_instruction=instruct_handler.dimasr['eos_instruct'],
    is_train=False
)

print(f"Dev samples to predict: {len(dev_formatted)}")

Dev samples to predict: 275


In [22]:
# Create loader for dev predictions
loader_dev = DatasetLoader(train_df_id=dev_formatted, test_df_id=None)
dev_ds, dev_tokenized_ds, _, _ = loader_dev.set_data_for_training_semeval_with_validation(
    t5_exp.tokenize_function_inputs
)

# Get predictions
dev_predictions = t5_exp.get_labels(dev_tokenized_ds, sample_set='train', batch_size=16)
print(f"Generated {len(dev_predictions)} predictions for dev set")

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Model loaded to:  cuda


100%|██████████| 18/18 [00:02<00:00,  6.31it/s]

Generated 275 predictions for dev set


In [23]:
# Format output as required JSONL
def format_output_jsonl(dev_formatted_df, predictions, output_path, t5_model):
    """
    Format predictions into required output JSONL format.
    
    Output format:
    {"ID": "...", "Aspect_VA": [{"Aspect": "...", "VA": "X.XX#X.XX"}, ...]}
    """
    # Group predictions by ID
    results = {}
    
    for idx, row in dev_formatted_df.iterrows():
        record_id = row['ID']
        aspect = row['aspect']
        va_pred = t5_model.parse_va_prediction(predictions[idx])
        
        if record_id not in results:
            results[record_id] = []
        
        results[record_id].append({
            'Aspect': aspect,
            'VA': va_pred
        })
    
    # Write output JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for record_id, aspect_vas in results.items():
            output_record = {
                'ID': record_id,
                'Aspect_VA': aspect_vas
            }
            f.write(json.dumps(output_record, ensure_ascii=False) + '\n')
    
    print(f"Output saved to: {output_path}")
    return results

# Generate output file
output_path = './predictions/pred_eng_laptop.jsonl'
results = format_output_jsonl(dev_formatted, dev_predictions, output_path, t5_exp)

Output saved to: ./predictions/pred_eng_laptop.jsonl


In [24]:
# Display sample predictions
print("\nSample predictions for submission:")
print("-" * 60)
for i, (record_id, aspect_vas) in enumerate(list(results.items())[:5]):
    print(f"\nID: {record_id}")
    for av in aspect_vas:
        print(f"  Aspect: {av['Aspect']:20} VA: {av['VA']}")


Sample predictions for submission:
------------------------------------------------------------

ID: lap26_aspect_va_dev_1
  Aspect: touchscreen          VA: 7.50#7.67

ID: lap26_aspect_va_dev_2
  Aspect: HP                   VA: 7.50#7.67

ID: lap26_aspect_va_dev_3
  Aspect: keyboard             VA: 7.50#7.67

ID: lap26_aspect_va_dev_4
  Aspect: screen size          VA: 7.50#7.67

ID: lap26_aspect_va_dev_5
  Aspect: Lenovo               VA: 2.50#7.62


In [25]:
# Verify output file format
print("\nFirst 3 lines of output file:")
print("-" * 60)
with open(output_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())


First 3 lines of output file:
------------------------------------------------------------
{"ID": "lap26_aspect_va_dev_1", "Aspect_VA": [{"Aspect": "touchscreen", "VA": "7.50#7.67"}]}
{"ID": "lap26_aspect_va_dev_2", "Aspect_VA": [{"Aspect": "HP", "VA": "7.50#7.67"}]}
{"ID": "lap26_aspect_va_dev_3", "Aspect_VA": [{"Aspect": "keyboard", "VA": "7.50#7.67"}]}


## Summary

This notebook has:
1. Loaded the DimASR training data (JSONL format with Quadruplet annotations)
2. Formatted the data with instruction prompts for the T5 model
3. Trained the model to generate valence#arousal scores
4. Evaluated on validation set using official RMSE_VA metric
5. Generated predictions for the dev set in submission format

The output file `predictions/pred_eng_laptop.jsonl` is ready for submission.